# 💰 Financial Performance & Profitability Analyzer
### P&L Analysis, Cost Center Identification & Forecasting | SQL + Python + Power BI

---

## 📖 The Business Story

A mid-size logistics company had a profitability problem — **revenue was growing at 8% YoY, but net margin was shrinking**. Leadership knew something was wrong but couldn't pinpoint where.

**As the Data Analyst, I was asked to:**
1. Analyze P&L data across business units and cost centers
2. Identify where margin erosion was happening and why
3. Build a financial dashboard that told the story — not just *what* happened, but *what it means* and *what to do*
4. Forecast the next two quarters using historical trends


## Step 1: Install & Import Libraries

In [ ]:
# !pip install pandas numpy matplotlib seaborn statsmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
BLUE   = "#1F4E79"
ORANGE = "#C55A11"
GREEN  = "#375623"
GRAY   = "#AAAAAA"

print("✅ Libraries loaded")


## Step 2: Generate Mock P&L Data

In [ ]:
np.random.seed(7)

MONTHS = pd.date_range("2022-01-01", "2023-12-31", freq="MS")
BUSINESS_UNITS = ["Northeast Dist.", "Southeast Dist.", "Mid-Atlantic Dist.", "Midwest Dist."]
COST_CENTERS = {
    "Labor - Regular":  {"budget_base": 280000, "overrun": 1.00},
    "Labor - Overtime": {"budget_base":  45000, "overrun": 1.38},  # ⚠️ problem
    "Freight":          {"budget_base": 190000, "overrun": 1.31},  # ⚠️ problem
    "Facilities":       {"budget_base":  95000, "overrun": 1.12},  # ⚠️ problem
    "Technology":       {"budget_base":  40000, "overrun": 0.97},
    "SG&A":             {"budget_base": 130000, "overrun": 1.11},
    "Maintenance":      {"budget_base":  30000, "overrun": 0.99},
    "Other OpEx":       {"budget_base":  25000, "overrun": 1.02},
}

pl_records, budget_records = [], []

for month in MONTHS:
    for bu in BUSINESS_UNITS:
        yoy      = 1.08 if month.year == 2023 else 1.0
        seasonal = 1 + 0.12 * np.sin((month.month - 3) * np.pi / 6)
        revenue  = 1_200_000 * yoy * seasonal * np.random.normal(1.0, 0.03)
        cogs     = revenue * 0.62 * (1.13 if month.year == 2023 else 1.0) * np.random.normal(1.0, 0.02)
        gp       = revenue - cogs
        opex     = sum(
            cfg["budget_base"] * cfg["overrun"] *
            (1.25 if cc in ["Labor - Overtime","Freight"] and month.month >= 7 else 1.0) *
            np.random.normal(1.0, 0.04)
            for cc, cfg in COST_CENTERS.items()
        )
        ebitda     = gp - opex
        net_income = ebitda * np.random.normal(0.82, 0.02)

        pl_records.append({
            "month": month.strftime("%Y-%m"), "year": month.year,
            "quarter": f"Q{month.quarter}", "business_unit": bu,
            "revenue": round(revenue,0), "cogs": round(cogs,0),
            "gross_profit": round(gp,0),
            "gross_margin_pct": round(gp/revenue*100, 2),
            "total_opex": round(opex,0), "ebitda": round(ebitda,0),
            "ebitda_margin_pct": round(ebitda/revenue*100, 2),
            "net_income": round(net_income,0),
            "net_margin_pct": round(net_income/revenue*100, 2),
        })

        for cc, cfg in COST_CENTERS.items():
            budget = cfg["budget_base"] * np.random.normal(1.0, 0.01)
            actual = budget * cfg["overrun"] * np.random.normal(1.0, 0.04)
            if cc in ["Labor - Overtime","Freight"] and month.month >= 7:
                actual *= 1.25
            budget_records.append({
                "month": month.strftime("%Y-%m"), "year": month.year,
                "quarter": f"Q{month.quarter}", "business_unit": bu,
                "cost_center": cc, "budget": round(budget,0),
                "actual": round(actual,0),
                "variance": round(actual-budget,0),
                "variance_pct": round((actual-budget)/budget*100, 2),
                "over_budget": actual > budget,
            })

pl  = pd.DataFrame(pl_records)
bud = pd.DataFrame(budget_records)

print(f"✅ P&L: {len(pl)} rows | Budget: {len(bud)} rows")
pl.head(3)


## Step 3: Year-over-Year P&L Comparison

Revenue is growing — so why is profitability declining?

In [ ]:
yearly = pl.groupby("year").agg(
    revenue      =("revenue","sum"),
    cogs         =("cogs","sum"),
    gross_profit =("gross_profit","sum"),
    total_opex   =("total_opex","sum"),
    ebitda       =("ebitda","sum"),
    net_income   =("net_income","sum"),
).reset_index()

yearly["gross_margin_pct"] = (yearly["gross_profit"] / yearly["revenue"] * 100).round(2)
yearly["net_margin_pct"]   = (yearly["net_income"]   / yearly["revenue"] * 100).round(2)
yearly["cogs_ratio"]       = (yearly["cogs"]         / yearly["revenue"] * 100).round(2)

for _, row in yearly.iterrows():
    print(f"\n{int(row['year'])}:")
    print(f"  Revenue:       ${row['revenue']/1e6:.2f}M")
    print(f"  Gross Margin:  {row['gross_margin_pct']:.1f}%")
    print(f"  Net Margin:    {row['net_margin_pct']:.1f}%")
    print(f"  COGS Ratio:    {row['cogs_ratio']:.1f}%")

yoy_rev    = (yearly.iloc[1]["revenue"]        / yearly.iloc[0]["revenue"] - 1) * 100
yoy_cogs   = (yearly.iloc[1]["cogs"]           / yearly.iloc[0]["cogs"]   - 1) * 100
margin_chg =  yearly.iloc[1]["gross_margin_pct"] - yearly.iloc[0]["gross_margin_pct"]

print(f"\n⚠️  Revenue grew {yoy_rev:.1f}% but COGS grew {yoy_cogs:.1f}%")
print(f"⚠️  Gross margin declined {abs(margin_chg):.1f} percentage points")


## Step 4: Margin Erosion Over Time

When did the margin start slipping?

In [ ]:
monthly_agg = pl.groupby("month").agg(
    revenue      =("revenue","sum"),
    gross_profit =("gross_profit","sum"),
    net_income   =("net_income","sum"),
    cogs         =("cogs","sum"),
).reset_index()

monthly_agg["gross_margin_pct"] = (monthly_agg["gross_profit"] / monthly_agg["revenue"] * 100)
monthly_agg["net_margin_pct"]   = (monthly_agg["net_income"]   / monthly_agg["revenue"] * 100)
monthly_agg["month_dt"]         = pd.to_datetime(monthly_agg["month"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_agg["month_dt"], monthly_agg["gross_margin_pct"],
        color=BLUE, linewidth=2.5, marker="o", markersize=3, label="Gross Margin %")
ax.plot(monthly_agg["month_dt"], monthly_agg["net_margin_pct"],
        color=ORANGE, linewidth=2.5, marker="o", markersize=3, label="Net Margin %")

# Highlight 2023
ax.axvline(pd.Timestamp("2023-01-01"), color=GRAY, linestyle="--", linewidth=1.5, label="2023 Start")
ax.fill_betweenx([monthly_agg["net_margin_pct"].min()-1, monthly_agg["gross_margin_pct"].max()+1],
                  pd.Timestamp("2023-01-01"), monthly_agg["month_dt"].max(),
                  alpha=0.05, color=ORANGE)

ax.set_title("Margin Erosion Over Time", fontsize=13, fontweight="bold", color=BLUE)
ax.set_ylabel("Margin (%)")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("\n🔍 Margin decline accelerates after Jan 2023 — coincides with COGS growth outpacing revenue")


## Step 5: Cost Center Variance Analysis

Which cost centers are driving the overrun?

In [ ]:
cc_2023 = bud[bud["year"] == 2023].groupby("cost_center").agg(
    budget   =("budget","sum"),
    actual   =("actual","sum"),
    variance =("variance","sum"),
).reset_index()
cc_2023["variance_pct"] = ((cc_2023["actual"] - cc_2023["budget"]) / cc_2023["budget"] * 100).round(1)
cc_2023 = cc_2023.sort_values("variance_pct", ascending=True)

colors = [ORANGE if v > 0 else BLUE for v in cc_2023["variance_pct"]]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("2023 Cost Center — Budget vs. Actual", fontsize=13, fontweight="bold", color=BLUE)

axes[0].barh(cc_2023["cost_center"], cc_2023["variance_pct"], color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Variance % vs. Budget", fontsize=11, color=BLUE)
axes[0].set_xlabel("Variance (%)")
for i, (_, row) in enumerate(cc_2023.iterrows()):
    axes[0].text(row["variance_pct"] + 0.3, i, f"{row['variance_pct']:+.1f}%", va="center", fontsize=8)

axes[1].barh(cc_2023["cost_center"], cc_2023["variance"]/1000, color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Variance $ (thousands)", fontsize=11, color=BLUE)
axes[1].set_xlabel("Variance ($K)")

plt.tight_layout()
plt.show()

over = cc_2023[cc_2023["variance_pct"] > 0]
print(f"\n⚠️  {len(over)} cost centers over budget:")
for _, row in over.sort_values("variance_pct", ascending=False).iterrows():
    print(f"   {row['cost_center']:25s}  +{row['variance_pct']:.1f}%  (${row['variance']/1e3:,.0f}K over)")


## Step 6: 2-Quarter Revenue Forecast

In [ ]:
from datetime import datetime

# Build monthly totals
monthly_agg = monthly_agg.sort_values("month_dt").reset_index(drop=True)

# Simple Holt-Winters or linear fallback
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    model = ExponentialSmoothing(monthly_agg["revenue"], trend="add",
                                  seasonal="add", seasonal_periods=12).fit(optimized=True)
    fc_revenue = model.forecast(6).values
    method = "Holt-Winters Exponential Smoothing"
except Exception:
    x = np.arange(len(monthly_agg))
    coeffs = np.polyfit(x, monthly_agg["revenue"].values, 1)
    fc_revenue = np.polyval(coeffs, np.arange(len(monthly_agg), len(monthly_agg)+6))
    method = "Linear Trend (fallback)"

future_months = pd.date_range(monthly_agg["month_dt"].iloc[-1] + pd.DateOffset(months=1),
                               periods=6, freq="MS")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_agg["month_dt"], monthly_agg["revenue"]/1e6,
        color=BLUE, linewidth=2.5, label="Actual Revenue")
ax.plot(future_months, fc_revenue/1e6,
        color=BLUE, linewidth=2.5, linestyle="--", label="Forecast")
ax.fill_between(future_months, fc_revenue*0.95/1e6, fc_revenue*1.05/1e6,
                alpha=0.15, color=BLUE, label="±5% confidence band")
ax.axvline(monthly_agg["month_dt"].iloc[-1], color=GRAY, linestyle=":", linewidth=1.5)
ax.set_title(f"2-Quarter Revenue Forecast ({method})", fontsize=12, fontweight="bold", color=BLUE)
ax.set_ylabel("Revenue ($M)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:.1f}M"))
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"\n📈 Projected Q1 2024 Revenue: ${fc_revenue[:3].sum()/1e6:.2f}M")
print(f"📈 Projected Q2 2024 Revenue: ${fc_revenue[3:].sum()/1e6:.2f}M")


## Step 7: Findings & Recommendations

### 🔴 Root Causes of Margin Erosion

| Finding | Detail | Estimated Annual Impact |
|---|---|---|
| COGS grew 13% vs 8% revenue | Supplier costs + spot freight rates | ~$1.2M gross profit erosion |
| Freight 31% over budget | Over-reliance on spot rates during Q3-Q4 | $420K avoidable cost |
| Overtime 38% over budget | Understaffing + demand spike, not planned | $180K avoidable cost |
| SG&A growing faster than revenue | 11% growth vs 8% revenue | Drag on net margin |

### ✅ Recommendations

| Action | Projected Savings |
|---|---|
| Renegotiate carrier contracts, reduce spot rate exposure | $420K annually |
| Demand-based labor scheduling model | $180K overtime reduction |
| Freeze non-essential SG&A until margin recovers | 1.5 pt margin recovery |
| Monthly cost center variance review with accountability | Early intervention on overruns |

> 💡 **Power BI Dashboard** on top of this analysis includes a P&L waterfall chart, budget vs. actual by cost center with conditional formatting, a rolling margin trend, and a 2-quarter forecast visual. See `powerbi/dax_measures.md`.
